# Hybrid ANN + GA Model for Butterfly Pea Experiment
This notebook trains an Artificial Neural Network (ANN) to predict four dependent variables from the Butterfly Pea flower extraction experiment using Genetic Algorithm (GA) for weight optimization.

In [6]:
# Install required packages (uncomment if running for the first time)
# !pip install numpy pandas deap tensorflow scikit-learn

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from deap import base, creator, tools, algorithms
import random

2025-04-17 22:48:38.837047: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744910318.893197   63931 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744910318.911991   63931 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1744910319.030538   63931 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1744910319.030551   63931 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1744910319.030552   63931 computation_placer.cc:177] computation placer alr

## Load and Prepare Data

In [4]:
df = pd.read_csv("Butterfly_Pea_Experiment.csv")
X = df.iloc[:, 1:3].values  # independent variables
y = df.iloc[:, 3:].values   # dependent variables

# Normalize input and output
x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()
X_scaled = x_scaler.fit_transform(X)
y_scaled = y_scaler.fit_transform(y)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.1, random_state=42)

In [11]:
import joblib
joblib.dump(x_scaler, 'x_scaler.save')
joblib.dump(y_scaler, 'y_scaler.save')


['y_scaler.save']

## Define ANN Model with 3 Hidden Layers

In [9]:
INPUT_DIM = X.shape[1]
OUTPUT_DIM = y.shape[1]

model = Sequential([
    Dense(16, input_dim=INPUT_DIM, activation='relu'),
    Dense(12, activation='relu'),
    Dense(8, activation='relu'),
    Dense(OUTPUT_DIM)  # Linear output
])

model.build()
total_weights = sum([np.prod(w.shape) for w in model.get_weights()])

/home/franz/Documents/Food_pro_project/food_pro_env/lib/python3.11/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


## Set Up Genetic Algorithm to Optimize ANN Weights

In [10]:
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()
toolbox.register("attr_float", lambda: random.uniform(-1.0, 1.0))
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_float, n=total_weights)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def set_weights(model, individual):
    weights = model.get_weights()
    new_weights = []
    start = 0
    for w in weights:
        shape = w.shape
        size = np.prod(shape)
        new_w = np.array(individual[start:start + size]).reshape(shape)
        new_weights.append(new_w)
        start += size
    model.set_weights(new_weights)

def evaluate(individual):
    set_weights(model, individual)
    preds = model.predict(X_train, verbose=0)
    return np.mean((preds - y_train) ** 2),

toolbox.register("evaluate", evaluate)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=0.5, indpb=0.1)
toolbox.register("select", tools.selTournament, tournsize=3)

## Run the Genetic Algorithm

In [ ]:
POP_SIZE = 50
N_GEN = 40
CX_PROB = 0.7
MUT_PROB = 0.2

pop = toolbox.population(n=POP_SIZE)
algorithms.eaSimple(pop, toolbox, cxpb=CX_PROB, mutpb=MUT_PROB, ngen=N_GEN, verbose=True)

## Evaluate the Best Model on Test Set

In [12]:
best_ind = tools.selBest(pop, k=1)[0]
set_weights(model, best_ind)

preds_scaled = model.predict(X_test, verbose=0)
preds = y_scaler.inverse_transform(preds_scaled)
y_true = y_scaler.inverse_transform(y_test)

for i in range(len(y_true)):
    print(f"Input: {x_scaler.inverse_transform([X_test[i]])[0]}")
    print(f"Actual: {y_true[i]}, Predicted: {preds[i]}\n")

Input: [ 0.05 20.  ]
Actual: [  0.258  26.718  10.929 -50.   ], Predicted: [  0.3003114 120.40084     2.990434  -12.455285 ]

Input: [ 0.351 15.   ]
Actual: [  0.582  60.116   4.334 -10.31 ], Predicted: [ 0.40580332  6.9223785   6.3835573  10.509851  ]



In [13]:
model.save('butterfly_pea_model.h5')


In [14]:
model.save('butterfly_pea_model.keras')


In [4]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppresses INFO, WARNING, and ERROR logs from TensorFlow

import numpy as np
from tensorflow.keras.models import load_model
import tensorflow as tf
import joblib

# Optional: even more control to silence logs
tf.get_logger().setLevel('ERROR')

# Define your scalers here or load them if saved
# Example:
# import joblib
# x_scaler = joblib.load('x_scaler.pkl')
# y_scaler = joblib.load('y_scaler.pkl')

def predict_with_user_input():
    model = load_model('butterfly_pea_model.keras')
    x_scaler = joblib.load('x_scaler.save')
    y_scaler = joblib.load('y_scaler.save')
    print("🔍 Enter the experimental values to get predictions:")
    
    weight = float(input("Enter Weight of Butterfly Pea Flower (g/200mL): "))
    time = float(input("Enter Pasteurization Time (mins @ 80°C): "))
    
    input_scaled = x_scaler.transform([[weight, time]])
    prediction_scaled = model.predict(input_scaled, verbose=0)
    prediction = y_scaler.inverse_transform(prediction_scaled)[0]
    
    labels = ['Phenolic Content (μg GAE/100mL)',
              'Anthocyanin Content (mg/100mL)',
              'Antioxidant Activity (%)',
              'Color Index']
    
    print("\n📊 Predicted Output Values:")
    for i, value in enumerate(prediction):
        print(f"{labels[i]}: {value:.4f}")
    
    print("\n🎯 Please enter the lab values")
    user_classification = []
    for label in labels:
        category = input(f"Classification for {label}: ").lower()
        user_classification.append(category)
    
    print("\n✅ Final Prediction Summary:")
    for i in range(len(labels)):
        print(f"{labels[i]} → Predicted: {prediction[i]:.4f}, Classified as: {user_classification[i].capitalize()}")

# Call it
predict_with_user_input()


🔍 Enter the experimental values to get predictions:


Enter Weight of Butterfly Pea Flower (g/200mL):  200
Enter Pasteurization Time (mins @ 80°C):  80



📊 Predicted Output Values:
Phenolic Content (μg GAE/100mL): 157.2489
Anthocyanin Content (mg/100mL): -20512.4570
Antioxidant Activity (%): -1715.9486
Color Index: 9060.4199

🎯 Please enter the lab values


Classification for Phenolic Content (μg GAE/100mL):  157
Classification for Anthocyanin Content (mg/100mL):  -20512
Classification for Antioxidant Activity (%):  -1715
Classification for Color Index:  9060



✅ Final Prediction Summary:
Phenolic Content (μg GAE/100mL) → Predicted: 157.2489, Classified as: 157
Anthocyanin Content (mg/100mL) → Predicted: -20512.4570, Classified as: -20512
Antioxidant Activity (%) → Predicted: -1715.9486, Classified as: -1715
Color Index → Predicted: 9060.4199, Classified as: 9060
